# Revision Agustin

## Segmento 1 ingreso bases

In [54]:
import pandas as pd
import re
import unicodedata
from IPython.display import display

# =========================
# RUTAS
# =========================
ruta_radicados = r"./input/Radicados.xlsx"
ruta_firmados = r"./input/Firmados.xlsx"
ruta_actividades = r"./input/ActividadesActuales.xlsx"

# =========================
# CARGA
# =========================
df_radicados_raw = pd.read_excel(ruta_radicados)
df_firmados_raw = pd.read_excel(ruta_firmados, header=None)
df_actividades_raw = pd.read_excel(ruta_actividades)

print("Base RADICADOS cargada:", df_radicados_raw.shape)
print("Base FIRMADOS cargada:", df_firmados_raw.shape)
print("Base ACTIVIDADES cargada:", df_actividades_raw.shape)

Base RADICADOS cargada: (2253, 25)
Base FIRMADOS cargada: (2271, 9)
Base ACTIVIDADES cargada: (2057, 22)


## Segmento 2 — Limpieza de títulos y visualización de bases

In [55]:
    # =========================
# FUNCIONES DE LIMPIEZA
# =========================
def quitar_tildes(texto):
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto

def limpiar_titulo(texto):
    texto = str(texto).strip()
    texto = quitar_tildes(texto)
    texto = texto.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

def columnas_unicas(cols):
    usadas = {}
    salida = []
    for c in cols:
        if c not in usadas:
            usadas[c] = 0
            salida.append(c)
        else:
            usadas[c] += 1
            salida.append(f"{c}_{usadas[c]}")
    return salida


# =========================
# 1. RADICADOS
# =========================
df_radicados = df_radicados_raw.copy()

columnas_radicados = [limpiar_titulo(c) for c in df_radicados.columns]
columnas_radicados = columnas_unicas(columnas_radicados)
df_radicados.columns = columnas_radicados

# quitar columnas basura unnamed
df_radicados = df_radicados.loc[:, ~df_radicados.columns.str.startswith("unnamed")]

# quitar primera fila informativa
df_radicados = df_radicados.iloc[1:].reset_index(drop=True)


# =========================
# 2. FIRMADOS
# =========================
df_firmados = df_firmados_raw.copy()

# el encabezado real está en la fila 3
nuevos_titulos_firmados = df_firmados.iloc[3].tolist()
nuevos_titulos_firmados = [limpiar_titulo(c) for c in nuevos_titulos_firmados]
nuevos_titulos_firmados = columnas_unicas(nuevos_titulos_firmados)

df_firmados = df_firmados.iloc[4:].reset_index(drop=True)
df_firmados.columns = nuevos_titulos_firmados


# =========================
# 3. ACTIVIDADES ACTUALES
# =========================
df_actividades = df_actividades_raw.copy()

# la fila 0 completa los nombres reales
fila0 = df_actividades.iloc[0]

nuevos_titulos_actividades = []
for col in df_actividades.columns:
    if str(col).startswith("Unnamed"):
        nuevos_titulos_actividades.append(fila0[col])
    else:
        # si ya tiene nombre útil, lo dejamos
        nuevos_titulos_actividades.append(col)

nuevos_titulos_actividades = [limpiar_titulo(c) for c in nuevos_titulos_actividades]
nuevos_titulos_actividades = columnas_unicas(nuevos_titulos_actividades)

df_actividades = df_actividades.iloc[1:].reset_index(drop=True)
df_actividades.columns = nuevos_titulos_actividades


# =========================
# MOSTRAR TITULOS LIMPIOS
# =========================
print("\n========== TITULOS RADICADOS ==========")
print(df_radicados.columns.tolist())
display(df_radicados.head(5))

print("\n========== TITULOS FIRMADOS ==========")
print(df_firmados.columns.tolist())
display(df_firmados.head(5))

print("\n========== TITULOS ACTIVIDADES ==========")
print(df_actividades.columns.tolist())
display(df_actividades.head(5))


========== TITULOS RADICADOS ==========
['no_doc', 'fecha_correo', 'radicado', 'tipo', 'f_radicado', 'asunto', 'titulo', 'elaboro', 'reviso', 'radico', 'of_radica', 'estado', 'medio', 'tramite', 'destinatarios', 'remitente', 'responsable', 'pre_radicado', 'tipo_asunto', 'registropor', 'ver', 'info_sae', 'estado_terminos']


,no_doc,fecha_correo,radicado,tipo,f_radicado,asunto,titulo,elaboro,reviso,radico,...,tramite,destinatarios,remitente,responsable,pre_radicado,tipo_asunto,registropor,ver,info_sae,estado_terminos
0,NaN,2026-01-02 07:44:28.317,20263000014,Memorando,2026-01-02 07:44:28.300,observaciones al proyecto de acto administrati...,N.A,Jhoanna Alexandra Montoya Muñoz,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Dirección Regional Tequendama,Dirección Jurídica Ambiental (Emma Constanza Z...,Argelio David Vargas Fragoz[Dirección Regional...,NaN,Devolucion de Acto Administrativo,ezunigag,NaN,NaN,NaN
1,NaN,2026-01-02 09:22:35.480,20263000020,Memorando,2026-01-02 09:22:35.413,Respuesta al radicado 06253001741: Respuesta a...,N.A,Juan Jacobo Finkielsztein Perez,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Dirección Regional Gualivá,Dirección Jurídica Ambiental (Emma Constanza Z...,Aura Daniela Leon Perez[Dirección Regional Gua...,NaN,NaN,ezunigag,NaN,NaN,NaN
2,NaN,2026-01-02 09:55:54.050,20263000021,Memorando,2026-01-02 09:55:54.013,Respuesta al radicado 20253123863: Reporte Pub...,N.A,Barbara Maria Ospina Acevedo,Juan Jacobo Finkielsztein Perez / DJA,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Secretaría General,Dirección Jurídica Ambiental (Emma Constanza Z...,Yadid Jimena Ariza Diaz[Secretaría General],NaN,Respuesta al memorando,ezunigag,NaN,NaN,NaN
3,NaN,2026-01-02 11:26:30.767,20263000031,Memorando,2026-01-02 11:26:30.680,Informes Técnicos Por Acoger Meta 12.2.5- Indi...,N.A,Hernan Mauricio Medina Forero,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,VARIOS (15 destinos),Dirección Jurídica Ambiental (Emma Constanza Z...,Alfonso Beltran Correa[Dirección Regional Gual...,NaN,NaN,ezunigag,NaN,NaN,NaN
4,NaN,2026-01-02 11:26:30.767,20263000031,Memorando,2026-01-02 11:26:30.680,Informes Técnicos Por Acoger Meta 12.2.5- Indi...,N.A,Hernan Mauricio Medina Forero,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,VARIOS (15 destinos),Dirección Jurídica Ambiental (Emma Constanza Z...,Diego Javier Carrillo Betancourt[Dirección Reg...,NaN,NaN,ezunigag,NaN,NaN,NaN



========== TITULOS FIRMADOS ==========
['expediente', 'tipo_tramite', 'tramite', 'estado_31dic2025', 'etapa_actual', 'regional', 'fecha_apertura', 'ano_creacion', 'estado_31mar']


,expediente,tipo_tramite,tramite,estado_31dic2025,etapa_actual,regional,fecha_apertura,ano_creacion,estado_31mar
0,26551,Permisivo,Licencia Ambiental,En Trámite,Expedición de Acto Administrativo,Sabana Centro,2005-10-14 00:00:00,2005,Resuelto
1,29164,Permisivo,Permiso de vertimientos,En Trámite,Evaluación,Sabana Centro,2007-02-28 00:00:00,2007,Resuelto
2,33536,Sancionatorio,Afectación recurso Suelo,En Trámite,Etapa Decisoria,Sabana Centro,2009-04-16 10:38:00,2009,Resuelto
3,41221,Sancionatorio,Afectación recurso Suelo,En Trámite,Etapa de Conocimiento,Ubate,2012-08-17 15:01:01.850000,2012,Resuelto
4,47452,Permisivo,Reglamentación de corrientes de aguas de uso p...,En Trámite,Expedición de Acto Administrativo,Soacha,2014-09-12 14:48:47.307000,2014,Resuelto



========== TITULOS ACTIVIDADES ==========
['expediente', 'tipo_de_tramite', 'tramite', 'estado_actual', 'regional', 'pac', 'fecha_apertura', 'responsable', 'fecha_asignacion', 'estado_usuario', 'dias_responsable', 'formatoexp', 'etapa', 'actividad', 'sidcar', 'estado', 'devoluciones', 'radicado', 'remite', 'fecha_asignacion_1', 'dias_responsable_1', 'comentarios']


,expediente,tipo_de_tramite,tramite,estado_actual,regional,pac,fecha_apertura,responsable,fecha_asignacion,estado_usuario,...,etapa,actividad,sidcar,estado,devoluciones,radicado,remite,fecha_asignacion_1,dias_responsable_1,comentarios
0,112546,Sancionatorio,Afectación recurso Aire,En Trámite,Magdalena Centro,Actividad 12.2.1,2025-02-10 16:46:56.183000,LADY PAOLA GALINDO RAMIREZ (DJA),2026-04-07 12:59:43.260000,Activo,...,Etapa de Conocimiento,Verificación,713241,Devuelto para correcciones,1,NaN,Leidy Viviana Alvarado Torres (DJA),2026-04-03 14:33:19.663000,2,Se considera necesario revisar la norma presun...
1,71180,Sancionatorio,Afectación recurso Agua,En Trámite,Chiquinquira,Actividad 12.2.1,2018-09-21 09:26:04.033000,Angela Maria Bedoya Gonzalez (DJA),2026-04-07 12:55:28,Activo,...,Etapa Decisoria,Acto administrativo que decide de fondo,NaN,NaN,NaN,NaN,SANDRA YISEL BORBON RODRIGUEZ (DJA),2026-03-24 10:33:13,8,Buenas tards doctora Angela se devuelve el exp...
2,73575,Sancionatorio,Afectación recurso Suelo,En Trámite,Sabana Centro,Actividad 12.2.1,2019-02-07 10:22:54.580000,Lorena Del Pilar Riaño Garcia (DJA),2026-04-07 12:53:40.787000,Activo,...,Etapa Decisoria,Acto administrativo que decide de fondo,612594,Enviado para Revisión,7,NaN,Jeimy Tatiana Plata Rueda (DJA),2026-03-25 18:48:29.827000,7,Dra por favor revisar comentarios en rojo al i...
3,84798,Sancionatorio,Afectación recurso Suelo,En Trámite,Sumapaz,Actividad 12.2.1,2020-12-30 19:20:05.363000,Juan Carlos Cubillos Pineda (DJA),2026-04-07 12:50:57,Activo,...,Etapa de Conocimiento,Formulación de cargos,NaN,NaN,NaN,NaN,Rosmery Nuñez Varon (DRSU),2026-03-18 11:11:21,11,NaN
4,97257,Sancionatorio,Afectación recurso Flora,En Trámite,Sumapaz,Actividad 12.2.1,2023-01-06 15:51:11.570000,Juan Carlos Cubillos Pineda (DJA),2026-04-07 12:49:43,Activo,...,Etapa de Conocimiento,Formulación de cargos,NaN,NaN,NaN,NaN,Rosmery Nuñez Varon (DRSU),2026-04-07 12:48:27,0,NaN


# Construcción de nueva tabla desde Radicados

## Segmento 3

In [56]:
# ==========================================
# SEGMENTO 3 - FILTRO DEVOLUCIONES + EXTRACCIÓN EXPEDIENTE
# ==========================================

import pandas as pd
import re
from IPython.display import display

rad = df_radicados.copy()

# ------------------------------------------
# 1. FILTRO: todo lo relacionado con devolución
# ------------------------------------------
patron_devolucion = r"devoluc|devuelt|devuelta|devolver"

rad["tipo_asunto_texto"] = rad["tipo_asunto"].astype(str).str.lower().str.strip()

rad_devoluciones = rad[
    rad["tipo_asunto_texto"].str.contains(patron_devolucion, na=False, regex=True)
].copy()

# ------------------------------------------
# 2. EXTRACTOR ROBUSTO DE EXPEDIENTE
# ------------------------------------------
def extraer_expediente_desde_asunto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()

    patrones = [
        # expediente / expediente. / expediente:
        r'\bexpediente\b\s*[:#\-\./]?\s*([0-9]{3,})',

        # exped / exped.
        r'\bexped\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\.\s*([0-9]{3,})',

        # expte / expte.
        r'\bexpte\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexpte\.\s*([0-9]{3,})',

        # exp / exp.
        r'\bexp\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexp\.\s*([0-9]{3,})',

        # ex / ex.
        r'\bex\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bex\.\s*([0-9]{3,})',

        # cuando el número aparece más adelante dentro del texto
        r'\bexpediente\b.*?([0-9]{3,})',
        r'\bexped\b.*?([0-9]{3,})',
        r'\bexpte\b.*?([0-9]{3,})',
        r'\bexp\b.*?([0-9]{3,})',
        r'\bex\b.*?([0-9]{3,})',

        # fallback final: primer número largo
        r'([0-9]{4,})'
    ]

    for patron in patrones:
        match = re.search(patron, texto, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    return None

rad_devoluciones["expediente_extraido"] = rad_devoluciones["asunto"].apply(extraer_expediente_desde_asunto)

# ------------------------------------------
# 3. SEPARAR RESPONSABLE EN NOMBRE Y REGIONAL
# ------------------------------------------
def separar_responsable(valor):
    if pd.isna(valor):
        return pd.Series([None, None])

    texto = str(valor).strip()

    match = re.match(r'^(.*?)\[(.*?)\]\s*$', texto)
    if match:
        nombre = match.group(1).strip()
        regional = match.group(2).strip()
        return pd.Series([nombre, regional])

    return pd.Series([texto, None])

rad_devoluciones[["nombre_responsable", "regional_responsable"]] = rad_devoluciones["responsable"].apply(separar_responsable)

print("Registros filtrados por devolución:", rad_devoluciones.shape)
display(
    rad_devoluciones[
        ["tipo_asunto", "asunto", "expediente_extraido", "responsable", "nombre_responsable", "regional_responsable"]
    ].head(20)
)

Registros filtrados por devolución: (238, 27)


,tipo_asunto,asunto,expediente_extraido,responsable,nombre_responsable,regional_responsable
0,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,66664,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama
23,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,106783,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
24,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,79807,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
27,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,58681,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá
44,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,24169,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente
89,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,78525,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz
90,Devolucion de Acto Administrativo,Devolución expediente 93132,93132,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita
148,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2454,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro
233,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,70277,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera
234,Devolucion de Acto Administrativo,Exp 15461. Devolución,15461,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro


## Segmento 4

In [57]:
# ==========================================
# SEGMENTO 4 - TABLA FINAL BASE + RESCATE DE EXPEDIENTE
# ==========================================

import re
from IPython.display import display

def rescatar_expediente_desde_asunto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()

    patrones_rescate = [
        r'\bexpediente\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\.\s*([0-9]{3,})',
        r'\bexpte\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexpte\.\s*([0-9]{3,})',
        r'\bexp\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexp\.\s*([0-9]{3,})',
        r'\bex\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bex\.\s*([0-9]{3,})',
        r'\bexpediente\b.*?([0-9]{3,})',
        r'\bexped\b.*?([0-9]{3,})',
        r'\bexpte\b.*?([0-9]{3,})',
        r'\bexp\b.*?([0-9]{3,})',
        r'\bex\b.*?([0-9]{3,})',
        r'([0-9]{4,})'
    ]

    for patron in patrones_rescate:
        match = re.search(patron, texto, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    return None


tabla_devoluciones_radicados = rad_devoluciones[
    [
        "expediente_extraido",
        "radicado",
        "tipo_asunto",
        "asunto",
        "fecha_correo",
        "f_radicado",
        "estado",
        "elaboro",
        "reviso",
        "responsable",
        "nombre_responsable",
        "regional_responsable"
    ]
].copy()

# renombrar estado de Radicados para no confundirlo
tabla_devoluciones_radicados = tabla_devoluciones_radicados.rename(
    columns={"estado": "estado_radicados"}
)

mask_nan = tabla_devoluciones_radicados["expediente_extraido"].isna()

tabla_devoluciones_radicados.loc[mask_nan, "expediente_extraido"] = (
    tabla_devoluciones_radicados.loc[mask_nan, "asunto"]
    .apply(rescatar_expediente_desde_asunto)
)

print("Tabla construida:", tabla_devoluciones_radicados.shape)
print("Expedientes vacíos después del rescate:",
      tabla_devoluciones_radicados["expediente_extraido"].isna().sum())

display(tabla_devoluciones_radicados.head(30))

Tabla construida: (238, 12)
Expedientes vacíos después del rescate: 0


,expediente_extraido,radicado,tipo_asunto,asunto,fecha_correo,f_radicado,estado_radicados,elaboro,reviso,responsable,nombre_responsable,regional_responsable
0,66664,20263000014,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama
23,106783,20263000220,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
24,79807,20263000222,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
27,58681,20263000225,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Finalizado,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá
44,24169,20263000446,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Finalizado,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente
89,78525,20263001018,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Finalizado,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz
90,93132,20263001072,Devolucion de Acto Administrativo,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Finalizado,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita
148,2454,20263001623,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Finalizado,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro
233,70277,20263002296,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,2026-01-13 14:50:27.597,2026-01-13 14:50:27.583,Finalizado,Johanna Isabel Muñoz Charry,Joaquin Fernando Sarmiento Cardenas / DJA,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera
234,15461,20263002297,Devolucion de Acto Administrativo,Exp 15461. Devolución,2026-01-13 14:51:49.670,2026-01-13 14:51:49.607,Finalizado,Diego Fernando Forero Gonzalez,NaN,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro


# Segmento 5 Revision y validacion de datos

In [58]:
# ==========================================
# SEGMENTO 5 - VALIDACIÓN EXPEDIENTES
# ==========================================

df = tabla_devoluciones_radicados.copy()

# total
total = len(df)

# con dato
con_expediente = df["expediente_extraido"].notna().sum()

# vacíos
sin_expediente = df["expediente_extraido"].isna().sum()

# porcentaje
porcentaje_completos = round((con_expediente / total) * 100, 2)

# tabla tipo dinámica
resumen_expedientes = pd.DataFrame({
    "total_registros": [total],
    "con_expediente": [con_expediente],
    "sin_expediente": [sin_expediente],
    "porcentaje_completos_%": [porcentaje_completos]
}) # ==========================================
# SEGMENTO 6 - LLAVE CONTRA ACTIVIDADESACTUALES
# ==========================================

import pandas as pd
import re
from IPython.display import display

def limpiar_llave_expediente(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    valor = re.sub(r"\.0$", "", valor)      # quita .0 si viene de numérico
    valor = re.sub(r"[^0-9]", "", valor)    # deja solo números
    return valor if valor != "" else None

# copia de trabajo
base_revision = tabla_devoluciones_radicados.copy()
base_actividades = df_actividades.copy()

# llaves limpias
base_revision["llave_expediente"] = base_revision["expediente_extraido"].apply(limpiar_llave_expediente)
base_actividades["llave_expediente"] = base_actividades["expediente"].apply(limpiar_llave_expediente)

# nos quedamos solo con las columnas necesarias desde ActividadesActuales
actividades_llave = base_actividades[["llave_expediente", "estado_actual"]].copy()

# por si una llave aparece repetida, dejamos la primera
actividades_llave = actividades_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])

# merge
base_revision = base_revision.merge(
    actividades_llave,
    on="llave_expediente",
    how="left"
)

# columnas de validación
base_revision["esta_en_actividadesactuales"] = base_revision["estado_actual"].notna().map({True: "SI", False: "NO"})
base_revision["estado_actividades"] = base_revision["estado_actual"]
base_revision["Estado_Agustin"] = base_revision["esta_en_actividadesactuales"].apply(
    lambda x: "Actividad en revision" if x == "SI" else ""
)

# quitamos columna auxiliar intermedia que ya quedó reflejada
base_revision = base_revision.drop(columns=["estado_actual"])

print("Cruce terminado:", base_revision.shape)

display(resumen_expedientes)

Cruce terminado: (238, 16)


,total_registros,con_expediente,sin_expediente,porcentaje_completos_%
0,238,238,0,100.0


# Crear llave y cruzar con ActividadesActuales

## Segmento 6

In [59]:
# ==========================================
# SEGMENTO 6 - LLAVE CONTRA ACTIVIDADESACTUALES
# ==========================================

import pandas as pd
import re
from IPython.display import display

def limpiar_llave_expediente(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    valor = re.sub(r"\.0$", "", valor)
    valor = re.sub(r"[^0-9]", "", valor)
    return valor if valor != "" else None

base_revision = tabla_devoluciones_radicados.copy()
base_actividades = df_actividades.copy()

base_revision["llave_expediente"] = base_revision["expediente_extraido"].apply(limpiar_llave_expediente)
base_actividades["llave_expediente"] = base_actividades["expediente"].apply(limpiar_llave_expediente)

columnas_actividades = ["llave_expediente"]
if "estado_actual" in base_actividades.columns:
    columnas_actividades.append("estado_actual")
if "remite" in base_actividades.columns:
    columnas_actividades.append("remite")

actividades_llave = base_actividades[columnas_actividades].copy()
actividades_llave = actividades_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])
actividades_llave["flag_actividades"] = "SI"

base_revision = base_revision.merge(
    actividades_llave,
    on="llave_expediente",
    how="left"
)

base_revision["esta_en_actividadesactuales"] = base_revision["flag_actividades"].fillna("NO")
base_revision["persona_que_asigna"] = base_revision["remite"] if "remite" in base_revision.columns else ""

columnas_borrar = [c for c in ["flag_actividades", "remite"] if c in base_revision.columns]
base_revision = base_revision.drop(columns=columnas_borrar)

print("Cruce con ActividadesActuales terminado:", base_revision.shape)

Cruce con ActividadesActuales terminado: (238, 16)


# Segmento 7 Dejar la base final y mostrar muestra de cómo quedó    

In [60]:
# ==========================================
# SEGMENTO 7 - BASE FINAL Y MUESTRA
# ==========================================

tabla_devoluciones_radicados = base_revision.copy()

columnas_finales = [
    "expediente_extraido",
    "llave_expediente",
    "radicado",
    "tipo_asunto",
    "asunto",
    "fecha_correo",
    "f_radicado",
    "estado_radicados",
    "elaboro",
    "reviso",
    "responsable",
    "nombre_responsable",
    "regional_responsable",
    "persona_que_asigna",
    "esta_en_actividadesactuales",
    "estado_actual",
    "Estado_Agustin"
]

columnas_finales = [c for c in columnas_finales if c in tabla_devoluciones_radicados.columns]
tabla_devoluciones_radicados = tabla_devoluciones_radicados[columnas_finales].copy()

print("Base final actualizada:", tabla_devoluciones_radicados.shape)
print("Columnas finales:")
print(tabla_devoluciones_radicados.columns.tolist())

display(tabla_devoluciones_radicados.head(10))

Base final actualizada: (238, 16)
Columnas finales:
['expediente_extraido', 'llave_expediente', 'radicado', 'tipo_asunto', 'asunto', 'fecha_correo', 'f_radicado', 'estado_radicados', 'elaboro', 'reviso', 'responsable', 'nombre_responsable', 'regional_responsable', 'persona_que_asigna', 'esta_en_actividadesactuales', 'estado_actual']


,expediente_extraido,llave_expediente,radicado,tipo_asunto,asunto,fecha_correo,f_radicado,estado_radicados,elaboro,reviso,responsable,nombre_responsable,regional_responsable,persona_que_asigna,esta_en_actividadesactuales,estado_actual
0,66664,66664,20263000014,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama,ADRIANA PAOLA RONDON GARCIA (DJA),SI,En Trámite
1,106783,106783,20263000220,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,NO,NaN
2,79807,79807,20263000222,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,NO,NaN
3,58681,58681,20263000225,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Finalizado,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá,NaN,NO,NaN
4,24169,24169,20263000446,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Finalizado,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente,NaN,NO,NaN
5,78525,78525,20263001018,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Finalizado,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz,NaN,NO,NaN
6,93132,93132,20263001072,Devolucion de Acto Administrativo,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Finalizado,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita,ANGI TATIANA CUERVO DAZA (DJA),SI,En Trámite
7,2454,2454,20263001623,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Finalizado,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro,NaN,NO,NaN
8,70277,70277,20263002296,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,2026-01-13 14:50:27.597,2026-01-13 14:50:27.583,Finalizado,Johanna Isabel Muñoz Charry,Joaquin Fernando Sarmiento Cardenas / DJA,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera,Joaquin Fernando Sarmiento Cardenas (DJA),SI,En Trámite
9,15461,15461,20263002297,Devolucion de Acto Administrativo,Exp 15461. Devolución,2026-01-13 14:51:49.670,2026-01-13 14:51:49.607,Finalizado,Diego Fernando Forero Gonzalez,NaN,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro,NaN,NO,NaN


# Analisis de base firmadoos

## Segmento 8 — Crear llave contra Firmados

In [61]:
# ==========================================
# SEGMENTO 8 - LLAVE CONTRA FIRMADOS
# ==========================================

base_final = tabla_devoluciones_radicados.copy()
base_firmados = df_firmados.copy()

base_firmados["llave_expediente"] = base_firmados["expediente"].apply(limpiar_llave_expediente)

columnas_firmados = ["llave_expediente"]
if "estado_31mar" in base_firmados.columns:
    columnas_firmados.append("estado_31mar")
if "estado_31dic2025" in base_firmados.columns:
    columnas_firmados.append("estado_31dic2025")

firmados_llave = base_firmados[columnas_firmados].copy()
firmados_llave = firmados_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])

if "estado_31mar" in firmados_llave.columns and "estado_31dic2025" in firmados_llave.columns:
    firmados_llave["estado_firmados"] = firmados_llave["estado_31mar"].combine_first(firmados_llave["estado_31dic2025"])
elif "estado_31mar" in firmados_llave.columns:
    firmados_llave["estado_firmados"] = firmados_llave["estado_31mar"]
elif "estado_31dic2025" in firmados_llave.columns:
    firmados_llave["estado_firmados"] = firmados_llave["estado_31dic2025"]
else:
    firmados_llave["estado_firmados"] = None

firmados_llave["flag_firmados"] = "SI"
firmados_llave = firmados_llave[["llave_expediente", "estado_firmados", "flag_firmados"]]

base_final = base_final.merge(
    firmados_llave,
    on="llave_expediente",
    how="left"
)

base_final["esta_en_firmados"] = base_final["flag_firmados"].fillna("NO")
base_final = base_final.drop(columns=["flag_firmados"])

print("Cruce con Firmados terminado:", base_final.shape)

Cruce con Firmados terminado: (238, 18)


## Segmento 9 — Actualizar columna de base y Estado_Agustin

In [62]:
# ==========================================
# SEGMENTO 9 - BASE ENCONTRADA + ESTADO_AGUSTIN
# ==========================================

def valor_no_vacio(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    if valor == "":
        return None
    if valor.lower() == "finalizado":
        return None
    return valor

def primer_valor_disponible(*args):
    for valor in args:
        limpio = valor_no_vacio(valor)
        if limpio is not None:
            return limpio
    return ""

def definir_base_encontrada(fila):
    en_actividades = fila.get("esta_en_actividadesactuales") == "SI"
    en_firmados = fila.get("esta_en_firmados") == "SI"

    if en_actividades and en_firmados:
        return "En Ambos"
    elif en_actividades:
        return "Actividades Actuales"
    elif en_firmados:
        return "En Firmados"
    else:
        return ""

base_final["estado_actividades"] = base_final.apply(
    lambda fila: primer_valor_disponible(
        fila.get("estado_actual"),
        fila.get("estado_radicados"),
        fila.get("estado_firmados")
    ),
    axis=1
)

base_final["esta_en_actividadesactuales"] = base_final.apply(definir_base_encontrada, axis=1)

def construir_estado_agustin(fila):
    estados = []

    if valor_no_vacio(fila.get("estado_actual")) is not None:
        estados.append("Actividad en revision")

    if fila.get("esta_en_firmados") == "SI":
        estados.append("Atendido firmado")

    return " | ".join(estados)

base_final["Estado_Agustin"] = base_final.apply(construir_estado_agustin, axis=1)

columnas_finales = [
    "expediente_extraido",
    "llave_expediente",
    "radicado",
    "tipo_asunto",
    "asunto",
    "fecha_correo",
    "f_radicado",
    "estado_radicados",
    "elaboro",
    "reviso",
    "responsable",
    "nombre_responsable",
    "regional_responsable",
    "persona_que_asigna",
    "esta_en_actividadesactuales",
    "estado_actividades",
    "esta_en_firmados",
    "Estado_Agustin"
]

columnas_finales = [c for c in columnas_finales if c in base_final.columns]
tabla_devoluciones_radicados = base_final[columnas_finales].copy()

print("Base actualizada:", tabla_devoluciones_radicados.shape)
display(tabla_devoluciones_radicados.head(20))

Base actualizada: (238, 18)


,expediente_extraido,llave_expediente,radicado,tipo_asunto,asunto,fecha_correo,f_radicado,estado_radicados,elaboro,reviso,responsable,nombre_responsable,regional_responsable,persona_que_asigna,esta_en_actividadesactuales,estado_actividades,esta_en_firmados,Estado_Agustin
0,66664,66664,20263000014,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama,ADRIANA PAOLA RONDON GARCIA (DJA),Actividades Actuales,En Trámite,NO,Actividad en revision
1,106783,106783,20263000220,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,En Firmados,Seguimiento y Control,SI,Atendido firmado
2,79807,79807,20263000222,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Finalizado,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,,,NO,
3,58681,58681,20263000225,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Finalizado,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá,NaN,,,NO,
4,24169,24169,20263000446,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Finalizado,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente,NaN,,,NO,
5,78525,78525,20263001018,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Finalizado,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz,NaN,,,NO,
6,93132,93132,20263001072,Devolucion de Acto Administrativo,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Finalizado,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita,ANGI TATIANA CUERVO DAZA (DJA),Actividades Actuales,En Trámite,NO,Actividad en revision
7,2454,2454,20263001623,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Finalizado,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro,NaN,,,NO,
8,70277,70277,20263002296,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,2026-01-13 14:50:27.597,2026-01-13 14:50:27.583,Finalizado,Johanna Isabel Muñoz Charry,Joaquin Fernando Sarmiento Cardenas / DJA,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera,Joaquin Fernando Sarmiento Cardenas (DJA),Actividades Actuales,En Trámite,NO,Actividad en revision
9,15461,15461,20263002297,Devolucion de Acto Administrativo,Exp 15461. Devolución,2026-01-13 14:51:49.670,2026-01-13 14:51:49.607,Finalizado,Diego Fernando Forero Gonzalez,NaN,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro,NaN,,,NO,


## Limpiar Datasets para entega de excel

In [63]:
# ==========================================
# LIMPIEZA PREVIA SOLO PARA EXPORTAR A EXCEL
# ==========================================

import pandas as pd

base_programa = tabla_devoluciones_radicados.copy()
df_exportar = tabla_devoluciones_radicados.copy()

# ------------------------------------------
# BASE ENCONTRADA PARA EXCEL
# ------------------------------------------
if 'esta_en_actividadesactuales' in df_exportar.columns:
    df_exportar['esta_en_actividadesactuales'] = (
        df_exportar['esta_en_actividadesactuales']
        .fillna('')
        .replace('SIN DATO', '')
    )
    df_exportar['base_encontrada'] = df_exportar['esta_en_actividadesactuales']

# ------------------------------------------
# LIMPIAR ESTADO_ACTIVIDADES
# ------------------------------------------
if 'estado_actividades' in df_exportar.columns:
    df_exportar['estado_actividades'] = df_exportar['estado_actividades'].apply(
        lambda x: '' if pd.notna(x) and str(x).strip().lower() == 'finalizado' else x
    )

# ------------------------------------------
# CÁLCULO DE DÍAS CORRIDOS Y ESTADO
# ------------------------------------------
if 'f_radicado' in df_exportar.columns:
    fecha_radicacion_tmp = pd.to_datetime(df_exportar['f_radicado'], errors='coerce')
    hoy = pd.Timestamp.today().normalize()

    df_exportar['dias de asignacion'] = (
        hoy - fecha_radicacion_tmp.dt.normalize()
    ).dt.days

    def clasificar_estado_devolucion(dias):
        if pd.isna(dias):
            return ""
        if dias >= 15:
            return "Vencido"
        elif dias >= 10:
            return "Por vencer"
        else:
            return "En termino"

    df_exportar['estado de devolucion'] = df_exportar['dias de asignacion'].apply(clasificar_estado_devolucion)

# ------------------------------------------
# ELIMINAR COLUMNAS AUXILIARES DE ENTREGA
# ------------------------------------------
columnas_quitar = [
    'llave_expediente',
    'tipo_asunto',
    'responsable',
    'esta_en_firmados',
    'esta_en_actividadesactuales',
    'estado_radicados',
    'fecha_correo'
]

columnas_quitar_reales = [c for c in columnas_quitar if c in df_exportar.columns]
df_exportar = df_exportar.drop(columns=columnas_quitar_reales)

# ------------------------------------------
# RENOMBRAR TÍTULOS FINALES
# ------------------------------------------
renombres = {
    'f_radicado': 'Fecha de radicacion',
    'nombre_responsable': 'Firmante',
    'Estado_Agustin': 'Estado Revision'
}

renombres_reales = {k: v for k, v in renombres.items() if k in df_exportar.columns}
df_exportar = df_exportar.rename(columns=renombres_reales)

# ------------------------------------------
# ORDEN FINAL DE SALIDA
# ------------------------------------------
orden_deseado = [
    'expediente_extraido',
    'radicado',
    'asunto',
    'Fecha de radicacion',
    'dias de asignacion',
    'estado de devolucion',
    'elaboro',
    'reviso',
    'Firmante',
    'regional_responsable',
    'persona_que_asigna',
    'base_encontrada',
    'estado_actividades',
    'Estado Revision'
]

orden_final = [c for c in orden_deseado if c in df_exportar.columns]
restantes = [c for c in df_exportar.columns if c not in orden_final]

df_exportar = df_exportar[orden_final + restantes].copy()

print("Dimensión base original:", base_programa.shape)
print("Dimensión base limpia para exportar:", df_exportar.shape)
display(df_exportar.head(8))

Dimensión base original: (238, 18)
Dimensión base limpia para exportar: (238, 14)


,expediente_extraido,radicado,asunto,Fecha de radicacion,dias de asignacion,estado de devolucion,elaboro,reviso,Firmante,regional_responsable,persona_que_asigna,base_encontrada,estado_actividades,Estado Revision
0,66664,20263000014,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.300,103,Vencido,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz,Dirección Regional Tequendama,ADRIANA PAOLA RONDON GARCIA (DJA),Actividades Actuales,En Trámite,Actividad en revision
1,106783,20263000220,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.447,100,Vencido,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,En Firmados,Seguimiento y Control,Atendido firmado
2,79807,20263000222,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.837,100,Vencido,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,,,
3,58681,20263000225,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.740,100,Vencido,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá,NaN,,,
4,24169,20263000446,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.053,99,Vencido,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente,NaN,,,
5,78525,20263001018,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.830,97,Vencido,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa,Dirección Regional Sumapaz,NaN,,,
6,93132,20263001072,Devolución expediente 93132,2026-01-08 10:37:51.677,97,Vencido,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita,ANGI TATIANA CUERVO DAZA (DJA),Actividades Actuales,En Trámite,Actividad en revision
7,2454,20263001623,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.217,96,Vencido,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro,NaN,,,


# Segmento prueba excel

In [64]:
# ==========================================
# EXPORTACIÓN FINAL (SOLO DATASET LIMPIO)
# ==========================================

import os

ruta_salida = r"C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\seguimiento_metas_wil\output"
archivo_viejo = "tabla_devoluciones_radicados.xlsx"
archivo_final = "tabla_devoluciones_radicados_FINAL.xlsx"

os.makedirs(ruta_salida, exist_ok=True)

ruta_vieja = os.path.join(ruta_salida, archivo_viejo)
ruta_final = os.path.join(ruta_salida, archivo_final)

# borrar archivo viejo si existe
if os.path.exists(ruta_vieja):
    try:
        os.remove(ruta_vieja)
    except Exception as e:
        print(f"No se pudo borrar el archivo viejo: {e}")

# exportar dataset final
df_exportar.to_excel(ruta_final, index=False)

print("✅ Archivo final generado correctamente.")
print("Ruta final:", ruta_final)
print("Dimensión exportada:", df_exportar.shape)

✅ Archivo final generado correctamente.
Ruta final: C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\seguimiento_metas_wil\output\tabla_devoluciones_radicados_FINAL.xlsx
Dimensión exportada: (238, 14)


# Tabla dinamica

In [68]:
# ==========================================
# TABLAS DINÁMICAS EJECUTIVAS PARA REPORTE
# ==========================================

import pandas as pd

df_reporte = df_exportar.copy()

# ------------------------------------------
# NORMALIZACIÓN REGIONAL (CLAVE)
# ------------------------------------------
if 'regional_responsable' in df_reporte.columns:
    df_reporte['regional_responsable'] = (
        df_reporte['regional_responsable']
        .fillna('Direccion Juridica Ambiental')
        .astype(str)
        .str.strip()
        .replace(['', 'SIN DATO'], 'Direccion Juridica Ambiental')
    )

# ------------------------------------------
# NORMALIZACIÓN CAMPOS
# ------------------------------------------
for col in ['Estado Revision', 'estado de devolucion', 'base_encontrada']:
    if col in df_reporte.columns:
        df_reporte[col] = (
            df_reporte[col]
            .fillna('SIN DATO')
            .astype(str)
            .str.strip()
            .replace('', 'SIN DATO')
        )

# ==========================================================
# TABLA 1: REGIONAL VS ESTADO REVISION
# ==========================================================
td_regional_revision = pd.crosstab(
    df_reporte['regional_responsable'],
    df_reporte['Estado Revision'],
    margins=True,
    margins_name='TOTAL'
).reset_index()

# ==========================================================
# TABLA 2: REGIONAL VS ESTADO DE DEVOLUCION
# ==========================================================
td_regional_devolucion = pd.crosstab(
    df_reporte['regional_responsable'],
    df_reporte['estado de devolucion'],
    margins=True,
    margins_name='TOTAL'
).reset_index()

# ==========================================================
# TABLA 3: CUANTOS HAY POR BASE_ENCONTRADA
# ==========================================================
td_base_encontrada = (
    df_reporte
    .groupby('base_encontrada', dropna=False)
    .size()
    .reset_index(name='cantidad')
    .sort_values('cantidad', ascending=False)
    .reset_index(drop=True)
)

# total general
total_general = pd.DataFrame({
    'base_encontrada': ['TOTAL'],
    'cantidad': [td_base_encontrada['cantidad'].sum()]
})

td_base_encontrada = pd.concat([td_base_encontrada, total_general], ignore_index=True)

# ==========================================================
# MOSTRAR RESULTADOS
# ==========================================================
print("\n==============================")
print("TABLA 1 - REGIONAL VS ESTADO REVISION")
print("==============================")
display(td_regional_revision)

print("\n==============================")
print("TABLA 2 - REGIONAL VS ESTADO DE DEVOLUCION")
print("==============================")
display(td_regional_devolucion)

print("\n==============================")
print("TABLA 3 - BASE ENCONTRADA (CUANTOS HAY)")
print("==============================")
display(td_base_encontrada)


TABLA 1 - REGIONAL VS ESTADO REVISION


Estado Revision,regional_responsable,Actividad en revision,Actividad en revision | Atendido firmado,Atendido firmado,SIN DATO,TOTAL
0,Direccion Juridica Ambiental,7,0,0,6,13
1,Dirección Jurídica Ambiental,8,0,0,4,12
2,Dirección Regional Almeidas y Guatavita,5,0,0,4,9
3,Dirección Regional Alto Magdalena,15,0,1,11,27
4,Dirección Regional Bajo Magdalena,2,0,0,0,2
5,Dirección Regional Bogotá – La Calera,6,0,0,12,18
6,Dirección Regional Chiquinquirá,5,0,0,6,11
7,Dirección Regional Gualivá,2,0,0,3,5
8,Dirección Regional Magdalena Centro,2,0,0,2,4
9,Dirección Regional Rionegro,4,0,0,5,9



TABLA 2 - REGIONAL VS ESTADO DE DEVOLUCION


estado de devolucion,regional_responsable,En termino,Por vencer,Vencido,TOTAL
0,Direccion Juridica Ambiental,13,0,0,13
1,Dirección Jurídica Ambiental,0,0,12,12
2,Dirección Regional Almeidas y Guatavita,0,0,9,9
3,Dirección Regional Alto Magdalena,0,0,27,27
4,Dirección Regional Bajo Magdalena,0,0,2,2
5,Dirección Regional Bogotá – La Calera,0,1,17,18
6,Dirección Regional Chiquinquirá,0,0,11,11
7,Dirección Regional Gualivá,0,1,4,5
8,Dirección Regional Magdalena Centro,0,1,3,4
9,Dirección Regional Rionegro,0,0,9,9



TABLA 3 - BASE ENCONTRADA (CUANTOS HAY)


,base_encontrada,cantidad
0,SIN DATO,152
1,Actividades Actuales,79
2,En Firmados,6
3,En Ambos,1
4,TOTAL,238
